In [1]:
import joblib

# Alur path default Kaggle: /kaggle/input/{nama-dataset-kamu}/{nama-file}
KAGGLE_PATH = '/kaggle/input/datasets/hanifsaifuddin/dataset-with-more-x-variabels/'

print("=== MEMUAT DATA & MODEL DARI INPUT KAGGLE ===")

# 1. Load Data Preprocessing
X_train_resampled = joblib.load(KAGGLE_PATH + 'X_train_resampled_values_v1.pkl')
y_train_resampled = joblib.load(KAGGLE_PATH + 'y_train_resampled_values_v1.pkl')
X_test = joblib.load(KAGGLE_PATH + 'X_test_values_v1.pkl')
y_test = joblib.load(KAGGLE_PATH + 'y_test_values_v1.pkl')

# 2. Load Base Learners yang sudah pintar dari lokal
# loaded_dt = joblib.load(KAGGLE_PATH + 'dt_model_v1.pkl')
# loaded_lr = joblib.load(KAGGLE_PATH + 'lr_model_v1.pkl')
# loaded_nb = joblib.load(KAGGLE_PATH + 'nb_model_v1.pkl')
# loaded_rf = joblib.load(KAGGLE_PATH + 'rf_model_v1.pkl')


print("Semua file biner sukses ditarik ke RAM cloud Kaggle! Siap gas Stacking.")

=== MEMUAT DATA & MODEL DARI INPUT KAGGLE ===
Semua file biner sukses ditarik ke RAM cloud Kaggle! Siap gas Stacking.


In [2]:
print(X_train_resampled)

[[2.67719800e+04 2.75210000e+04 5.42929800e+04 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [3.44879650e+05 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [1.86260784e+06 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 1.00000000e+00]
 ...
 [8.44213810e+05 8.44213810e+05 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 1.00000000e+00]
 [8.71893150e+05 8.71893150e+05 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [2.13532018e+06 2.13532018e+06 0.00000000e+00 ... 0.00000000e+00
  0.00000000e+00 1.00000000e+00]]


In [3]:
import joblib
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score,accuracy_score,recall_score,precision_score,f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

# (Pastikan baris load pkl data & model dasar dari input Kaggle path)

# Susun ulang tuple estimators heterogen untuk Stacking dasar
# estimators = [
#     ('dt', loaded_dt), ('lr', loaded_lr), ('nb', loaded_nb), ('rf', loaded_rf)
# ]
estimators = [
    ('dt', DecisionTreeClassifier(random_state=42)),
    ('lr', LogisticRegression(random_state=42, max_iter=1000)),
    ('nb', GaussianNB()),
    ('rf', RandomForestClassifier(random_state=42, n_jobs=-1))
    
]
# Definisikan eksperimen Stacking (Meta-Learner bergantian sesuai baris jurnal)
stacking_experiments = {
    'Stacking_DT': StackingClassifier(estimators=estimators, final_estimator=DecisionTreeClassifier(random_state=42), cv=5, n_jobs=-1),
    'Stacking_LR': StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(random_state=42, max_iter=1000), cv=5, n_jobs=-1),
    'Stacking_NB': StackingClassifier(estimators=estimators, final_estimator=GaussianNB(), cv=5, n_jobs=-1),
    'Stacking_RF': StackingClassifier(estimators=estimators, final_estimator=RandomForestClassifier(random_state=42, n_jobs=-1), cv=5, n_jobs=-1)
}

print("=== EKSPERIMEN STACKING (KAGGLE BACKGROUND) ===")
for name, model in stacking_experiments.items():
    model.fit(X_train_resampled, y_train_resampled)
    preds = model.predict(X_test)
    print(f"Model: {name}")
    print(classification_report(y_test, preds))

    print(f"=========== Hasil dari {name} =============\n")
    print(f"{name} Confusion Matrix: {confusion_matrix(y_test, preds)}\n")
    

    print(f"{name} Accuracy Score: {accuracy_score(y_test, preds):.4f}\n")
    

    print(f"{name} Recall Score: {recall_score(y_test, preds):.4f}\n")
   

    print(f"{name}Precision Score: {precision_score(y_test, preds):.4f}\n")
   
    print(f"{name} | F1-Score: {f1_score(y_test, preds):.4f}\n")
    
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        auc_score = roc_auc_score(y_test, y_proba)
        print(f"AUC-ROC Score: {auc_score:.4f}\n")
    
    
    joblib.dump(model, f'{name}_final.pkl')

=== EKSPERIMEN STACKING (KAGGLE BACKGROUND) ===
Model: Stacking_DT
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1906351
           1       1.00      0.61      0.76      2435

    accuracy                           1.00   1908786
   macro avg       1.00      0.80      0.88   1908786
weighted avg       1.00      1.00      1.00   1908786

=========== Hasil dari Stacking_DT =============

Stacking_DT Confusion Matrix: [[1906351       0]
 [    950    1485]]

Stacking_DT Accuracy Score: 0.9995

Stacking_DT Recall Score: 0.6099

Stacking_DTPrecision Score: 1.0000

Stacking_DT | F1-Score: 0.7577

AUC-ROC Score: 0.8049

Model: Stacking_LR
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1906351
           1       0.90      0.87      0.89      2435

    accuracy                           1.00   1908786
   macro avg       0.95      0.94      0.94   1908786
weighted avg       1.00      1.00 